# CS5720 Home Assignment 1

**Jeff Agnitsch** · Student ID: 7005146960 · Fall 2026

See `ROADMAP.md` for the phase-by-phase plan and concept notes.

In [1]:
# Shared imports and one fixed seed so every run (and the video) is reproducible
import os
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)      # seeds Python, NumPy, and TensorFlow together
os.makedirs("images", exist_ok=True)    # charts are saved here for the README

# Two fixed series colors used in every chart: blue = first series, orange = second
BLUE, ORANGE = "#2a78d6", "#eb6834"

print("TensorFlow", tf.__version__)
print("GPUs available:", len(tf.config.list_physical_devices("GPU")), "(CPU-only on native Windows)")

TensorFlow 2.21.0


GPUs available: 0 (CPU-only on native Windows)


## Task 1 — Tensor manipulation and reshaping

Goal: build a (4, 6) tensor, print rank/shape, reshape to (2, 3, 4), transpose to (3, 2, 4), broadcast a (1, 4) tensor onto it.

Why: shape bugs are the most common deep-learning bugs; this is the vocabulary for every layer that follows.

In [2]:
# --- Task 1: rank, shape, reshape, transpose, broadcast ---

def describe(name, x):
    """Print rank and shape using TensorFlow functions (tf.rank gives the number of axes)."""
    print(f"{name:<22} rank={int(tf.rank(x))}  shape={tuple(x.shape)}")

# 1. Random tensor of shape (4, 6), values uniform in [0, 1)
t = tf.random.uniform((4, 6))
describe("original", t)

# 2. Reshape to (2, 3, 4). Legal because 4*6 = 24 = 2*3*4.
#    Reshape keeps the elements in the same row-major order; it only regroups them.
r = tf.reshape(t, (2, 3, 4))
describe("reshaped (2,3,4)", r)

# 3. Transpose to (3, 2, 4). perm=[1, 0, 2] swaps axes 0 and 1 and keeps axis 2.
#    Unlike reshape, transpose actually moves data around.
p = tf.transpose(r, perm=[1, 0, 2])
describe("transposed (3,2,4)", p)

# 4. Broadcast a (1, 4) tensor onto (3, 2, 4) and add.
#    Aligning from the right: (1,4) -> padded to (1,1,4) -> stretched to (3,2,4).
small = tf.random.uniform((1, 4))
describe("small (1,4)", small)
added = p + small
describe("p + small", added)

# Prove the broadcast is the same as materialising the stretched tensor explicitly
explicit = tf.broadcast_to(small, p.shape)
print("broadcast_to shape:", tuple(explicit.shape),
      "| p + small == p + explicit:", bool(tf.reduce_all(added == p + explicit)))

# Counter-example: (1, 4) cannot broadcast against the ORIGINAL (4, 6) tensor,
# because the last axes are 4 vs 6 (neither equal nor 1).
try:
    t + small
except tf.errors.InvalidArgumentError as e:
    print("(4,6) + (1,4) fails as expected:", str(e).splitlines()[0])

original               rank=2  shape=(4, 6)
reshaped (2,3,4)       rank=3  shape=(2, 3, 4)
transposed (3,2,4)     rank=3  shape=(3, 2, 4)
small (1,4)            rank=2  shape=(1, 4)
p + small              rank=3  shape=(3, 2, 4)
broadcast_to shape: (3, 2, 4) | p + small == p + explicit: True
(4,6) + (1,4) fails as expected: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes: [4,6] vs. [1,4] [Op:AddV2] name: 


### How broadcasting works in TensorFlow

Broadcasting lets TensorFlow combine tensors of different shapes without copying data. The shapes are compared **from the rightmost axis leftward**:

1. If one tensor has fewer axes, its shape is padded on the **left** with 1s.
2. Two axes are compatible when they are **equal** or **one of them is 1**.
3. Every axis of size 1 is treated as if it were repeated to match the other tensor. No copy is made; the same value is read for every position along that axis.

For this task: `(1, 4)` against `(3, 2, 4)` becomes `(1, 1, 4)`, then `1→3`, `1→2`, `4=4`, so the result is `(3, 2, 4)`. The same `(1, 4)` tensor fails against the original `(4, 6)` because the last axes are 4 and 6, which are neither equal nor 1. The counter-example in the cell above shows that error.